# 01 · Data — manifest, image cache, and a backbone sanity check


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


## Which dataset?

| source | access | images | reports | labels |
|---|---|---|---|---|
| `mimic` | PhysioNet **credentialed** (CITI course + signed DUA) | 377k | yes | CheXpert labeler |
| `openi` | public, no account | ~7.4k | yes | MeSH terms |
| `synthetic` | none | generated | templated | exact |

MIMIC-CXR-JPG is the right dataset for this project. Getting access takes a few
days: complete the CITI "Data or Specimens Only Research" course, upload the
certificate to PhysioNet, then sign the MIMIC-CXR-JPG data use agreement.

**While you wait, use `openi`** — it is a real chest X-ray dataset with real
reports, it needs no credentials, and every later notebook works unchanged.

## What this notebook does not do

It does **not** download the full 570 GB release. It fetches three small
metadata tables, decides which `subset_size` studies to use, and then downloads
only those images.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found - you are on CPU. Runtime > Change runtime type > T4 GPU.")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

In [ ]:
# Everything below is overridden on the command line, so this cell is the only
# place you need to edit. Values here are tuned for a Colab T4.
CFG = dict(
    subset_size = 12000,     # frontal studies pulled from MIMIC-CXR-JPG
    text_mode   = 'indication',   # see docs/LEAKAGE.md before changing this
    batch_size  = 24,
    epochs      = 12,
    diffusion_epochs = 30,
    synth_per_class  = 500,
)

def dvlhg(command, **overrides):
    """Run a dvlhg subcommand with RUN_ROOT and any overrides applied."""
    import os, shlex, subprocess, sys
    args = [sys.executable, '-m', 'dvlhg.cli', *shlex.split(command)]
    args += ['--set', f"paths.root={os.environ['RUN_ROOT']}"]
    for key, value in overrides.items():
        args += ['--set', f'{key}={value}']
    print('$', ' '.join(args[2:]))
    return subprocess.run(args, check=True)

### Credentials

Typed into a password box, not stored in the notebook. Skip this cell entirely
if you are using `openi`.

In [ ]:
import getpass, os
SOURCE_DATASET = 'mimic'     # 'mimic' | 'openi' | 'synthetic'

if SOURCE_DATASET == 'mimic':
    os.environ['PHYSIONET_USER'] = input('PhysioNet username: ')
    os.environ['PHYSIONET_PASS'] = getpass.getpass('PhysioNet password: ')
    print('credentials set for this session only')

### Build the manifest and the image cache

This is the long step — downloading ~12k JPEGs takes roughly 20–40 minutes on
Colab. It is resumable: files already present are skipped, so a disconnect
costs nothing.

The image cache is written as **one** `.npy` memmap rather than 12k small
files, because copying 12k files to Drive takes the better part of an hour and
copying one 800 MB file takes two minutes.

In [ ]:
dvlhg('prep',
      **{'data.source': SOURCE_DATASET,
         'data.subset_size': CFG['subset_size'],
         'text.mode': CFG['text_mode']})

### What did we get?

In [ ]:
import json, os, pandas as pd
from dvlhg.config import load_config
from dvlhg.data.prepare import load_manifest

cfg = load_config('configs/default.yaml',
                  [f"paths.root={os.environ['RUN_ROOT']}",
                   f"data.source={SOURCE_DATASET}", f"text.mode={CFG['text_mode']}"])
manifest = load_manifest(cfg)
stats = json.load(open(os.path.join(cfg.paths.manifest, 'stats.json')))

print('rows:', len(manifest), '| splits:', stats['splits'])
print('text mode:', stats['text_mode'], '-', stats['text_mode_description'])
print('leaky:', stats['text_is_leaky'])
print()
print(pd.DataFrame(stats['label_stats_all']).T.round(3))
print()
print('example text the model will see:')
print(' ', manifest['text'].iloc[0][:300])

### Class balance and co-occurrence

The co-occurrence matrix is worth a look — it is the structure the hypergraph's
prototype hyperedges are meant to exploit.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from dvlhg.constants import LABELS
from dvlhg.data.labels import targets_from_manifest

y, mask = targets_from_manifest(manifest)
counts = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        both = (mask[:, i] > 0) & (mask[:, j] > 0)
        counts[i, j] = ((y[both, i] > .5) & (y[both, j] > .5)).sum()

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(counts, cmap='Blues')
ax.set_xticks(range(4)); ax.set_xticklabels(LABELS, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(LABELS, fontsize=8)
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(counts[i, j]), ha='center', va='center', fontsize=8,
                color='white' if counts[i, j] > counts.max()/2 else '#0b0b0b')
ax.set_title('co-occurrence of positive labels', fontsize=10, loc='left')
plt.tight_layout(); plt.show()

### Sanity check the backbone **before** spending GPU hours

`check-vlm` runs BiomedCLIP zero-shot: it scores each film against
"chest x-ray showing {finding}" versus "chest x-ray with no {finding}" and
reports AUROC. No training involved.

A correctly loaded BiomedCLIP lands somewhere around 0.6–0.8 here. If you see
~0.50, the weights did not download and you would be fine-tuning random
initialisation for an hour without noticing.

In [ ]:
dvlhg('check-vlm --limit 600',
      **{'data.source': SOURCE_DATASET, 'text.mode': CFG['text_mode']})

### Next

**02 · Diffusion.** If you want to skip the diffusion stage entirely, go
straight to **03** and pass `--set train.use_synthetic=false`.